In [10]:
import numpy as np 
import torch.nn.functional as F
import torch

Q = np.random.rand(1024, 128)
K = np.random.rand(1024, 128)
V = np.random.rand(1024, 128)

def softmax(x):
    max_vals = np.max(x, axis=-1, keepdims=True)
    top = np.exp(x - max_vals) 
    bottom = np.sum(np.exp(x - max_vals), keepdims=True, axis=-1)
    return top / bottom

In [18]:
def softmax_causal(x):
    m_dim, n_dim = x.shape 
    assert len(x.shape) == 2 and "only work for dim2 for now"
    result = np.zeros(x.shape)

    for m in range(m_dim):
        max_val = np.max(x[m, 0:(m+1)])
        sum_val = np.sum(np.exp(x[m, 0:(m+1)] - np.array([max_val])))
        for n in range(n_dim):
            if n <= m: 
                result[m, n] = np.exp(x[m, n] - max_val) / sum_val 
            else: 
                result[m, n] = 0
    return result

def pre_softmax(x):
    return x

def my_attention(Q, K, V): 
    dim_sequence, dim_model = Q.shape 
    _, dim_embedding = V.shape 
    output = np.zeros((dim_sequence, dim_embedding))
    
    first_gemm = pre_softmax(Q @ K.T)
    after_softmax = softmax_causal(first_gemm)
    output = after_softmax @ V
    return output

my_atten = my_attention(Q, K, V)
torch_atten = F.scaled_dot_product_attention(torch.from_numpy(Q), torch.from_numpy(K), torch.from_numpy(V), is_causal=True, scale=1)
print(f"My implementation:\n {my_atten}")
print(f"Pytorch implementation:\n {torch_atten}")
print(f"All close: {torch.allclose(torch.from_numpy(my_atten), torch_atten)}")

My implementation:
 [[0.90111487 0.61683101 0.78710242 ... 0.03860431 0.97076037 0.17282449]
 [0.89238434 0.60669325 0.74035742 ... 0.06716157 0.96291747 0.18713886]
 [0.80920392 0.63393797 0.50256405 ... 0.24044804 0.78165685 0.255813  ]
 ...
 [0.5897131  0.50439614 0.62784292 ... 0.52701563 0.47839188 0.59125262]
 [0.52011422 0.4642289  0.56647308 ... 0.50801259 0.5514078  0.50099205]
 [0.52435273 0.4860058  0.5715082  ... 0.50457683 0.52324914 0.512083  ]]
Pytorch implementation:
 tensor([[0.9011, 0.6168, 0.7871,  ..., 0.0386, 0.9708, 0.1728],
        [0.8924, 0.6067, 0.7404,  ..., 0.0672, 0.9629, 0.1871],
        [0.8092, 0.6339, 0.5026,  ..., 0.2404, 0.7817, 0.2558],
        ...,
        [0.5897, 0.5044, 0.6278,  ..., 0.5270, 0.4784, 0.5913],
        [0.5201, 0.4642, 0.5665,  ..., 0.5080, 0.5514, 0.5010],
        [0.5244, 0.4860, 0.5715,  ..., 0.5046, 0.5232, 0.5121]],
       dtype=torch.float64)
All close: True


In [76]:
my_atten.shape

(10, 128)